# 6.2. 图像卷积

卷积运算是卷积神经网络的核心操作。在本节中，我们将学习二维卷积的工作原理。

In [ ]:
import tensorflow as tf
import numpy as np

## 6.2.1. 互相关运算

严格来说，卷积层是一个互相关运算（cross-correlation），而不是数学上的卷积运算。

In [ ]:
def corr2d(X, K):
    """计算二维互相关运算"""
    h, w = K.shape
    Y = tf.Variable(tf.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1)))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j].assign(tf.reduce_sum(
                X[i: i + h, j: j + w] * K))
    return Y

## 6.2.2. 验证互相关运算

In [ ]:
# 构造一个输入张量X和一个卷积核张量K
X = tf.constant([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = tf.constant([[0.0, 1.0], [2.0, 3.0]])

print("输入 X:")
print(X.numpy())
print("\n卷积核 K:")
print(K.numpy())
print("\n输出 Y:")
print(corr2d(X, K).numpy())

## 6.2.3. 卷积层

卷积层对输入和卷积核权重进行互相关运算，并加上一个标量偏置来得到输出。

In [ ]:
class Conv2D(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__()

    def build(self, kernel_size):
        initializer = tf.random_normal_initializer()
        self.weight = self.add_weight(name='w', shape=kernel_size,
                                       initializer=initializer)
        self.bias = self.add_weight(name='b', shape=(1,),
                                     initializer=initializer)

    def call(self, inputs):
        return corr2d(inputs, self.weight) + self.bias

## 6.2.4. 边缘检测

卷积层的一个简单应用是通过找到像素变化的位置来检测图像中不同颜色的边缘。

In [ ]:
# 构造一个6×8像素的图像（黑白两个区域）
X = tf.Variable(tf.ones((6, 8)))
X[:, 2:6].assign(tf.zeros((6, 4)))

print("原始图像:")
print(X.numpy())

In [ ]:
# 构造一个高度为1、宽度为2的卷积核
K = tf.constant([[1.0, -1.0]])

# 执行互相关运算
Y = corr2d(X, K)
print("边缘检测结果:")
print(Y.numpy())
print("\n可以看到，1表示从白到黑的边缘，-1表示从黑到白的边缘")

## 6.2.5. 学习卷积核

我们可以通过数据来学习卷积核。

In [ ]:
# 构造一个卷积层
conv2d = tf.keras.layers.Conv2D(1, (1, 2), use_bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、高度、宽度、通道数）
X = tf.reshape(X, (1, 6, 8, 1))
Y = tf.reshape(Y, (1, 6, 7, 1))

# 学习率
lr = 3e-2

# 训练
Y_hat = conv2d(X)
for i in range(10):
    with tf.GradientTape(watch_accessed_variables=False) as g:
        g.watch(conv2d.weights[0])
        Y_hat = conv2d(X)
        l = (abs(Y_hat - Y)) ** 2
        # 更新卷积核
    update = tf.multiply(lr, g.gradient(l, conv2d.weights[0]))
    weights = conv2d.get_weights()
    weights[0] = conv2d.weights[0] - update
    conv2d.set_weights(weights)
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {tf.reduce_sum(l):.3f}')

In [ ]:
# 查看学到的卷积核
print("学习到的卷积核:")
print(tf.reshape(conv2d.weights[0], (1, 2)).numpy())
print("\n与真实卷积核 [1, -1] 非常接近！")

## 小结

1. 二维卷积层的核心计算是二维互相关运算
2. 卷积层可以用来检测图像的边缘
3. 卷积核的参数可以通过数据学习得到
4. 卷积层在处理图像时保留了空间结构